# One bounded trajectory model conditioned on particle number $N$

The recurrent input is $[s_z(t),t_{\mathrm{norm}},N/N_{\mathrm{ref}}]$. The model predicts a categorical next-state distribution on 401 grid points in $[-1,1]$.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

cwd = Path.cwd().resolve()
if (cwd / "withN_singleO").is_dir():
    project_dir = cwd
    experiment_dir = cwd / "withN_singleO"
elif cwd.name == "withN_singleO":
    project_dir = cwd.parent
    experiment_dir = cwd
else:
    raise RuntimeError("Start Jupyter from the project root or withN_singleO/.")

sys.path.insert(0, str(experiment_dir))

from dataset import MultiNNextStateDataset, stratified_split_indices
from model import ConditionedTrajectoryGRUGrid, interpolated_grid_nll
from train_grid_model import (
    combine_trajectory_groups,
    evaluate,
    generate_trajectories,
    load_trajectory_groups,
    rollout_summary,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project directory:", project_dir)
print("Using:", device)

In [ ]:
plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Configuration and data loading

For the final model, leave `held_out_N_values = []` to train on every available $N$. Before trusting extrapolation, set it to `[18]`: the model then trains only through $N=16$, so evaluation at $N=18$ mimics the first extrapolation step.

In [ ]:
all_N_values = [4, 6, 8, 10, 12, 14, 16, 18]
held_out_N_values = []  # Recommended diagnostic run: [18]
trajectories_per_N = 500
n_reference = 18.0
state_min = -1.0
state_max = 1.0
time_end = 1.0
seed = 42

candidate_data_roots = [
    project_dir / "GenerateTraj" / "data",
    project_dir / "generate_traj" / "data",
]
data_root = next(
    (path for path in candidate_data_roots if path.is_dir()),
    candidate_data_roots[0],
)

trajectory_groups, source_files = load_trajectory_groups(
    data_root,
    all_N_values,
    trajectories_per_n=trajectories_per_N,
)

print("Data root:", data_root.resolve())
for N in all_N_values:
    values = trajectory_groups[N]
    print(
        f"N={N:2d}: shape={values.shape}, "
        f"range=[{values.min():.4f}, {values.max():.4f}], "
        f"initial=[{values[:, 0].min():.4f}, {values[:, 0].max():.4f}]"
    )

## Balanced dataset and stratified split

Every $N$ contributes the same number of trajectories. Each $N$ is split independently into train, validation, and test subsets.

In [ ]:
fit_N_values = [N for N in all_N_values if N not in held_out_N_values]
if not fit_N_values:
    raise ValueError("At least one N value must remain for training.")

sz, particle_numbers = combine_trajectory_groups(
    trajectory_groups, fit_N_values
)
dataset = MultiNNextStateDataset(
    sz,
    particle_numbers,
    time_end=time_end,
    state_min=state_min,
    state_max=state_max,
    n_reference=n_reference,
)

train_indices, validation_indices, test_indices = stratified_split_indices(
    particle_numbers,
    train_fraction=0.8,
    validation_fraction=0.1,
    seed=seed,
)
train_set = Subset(dataset, train_indices)
validation_set = Subset(dataset, validation_indices)
test_set = Subset(dataset, test_indices)

batch_size = 64
loader_options = {
    "batch_size": batch_size,
    "pin_memory": device.type == "cuda",
}
train_loader = DataLoader(
    train_set,
    shuffle=True,
    generator=torch.Generator().manual_seed(seed),
    **loader_options,
)
validation_loader = DataLoader(
    validation_set, shuffle=False, **loader_options
)
test_loader = DataLoader(test_set, shuffle=False, **loader_options)

features, targets = next(iter(train_loader))
print("Fit N values:", fit_N_values)
print("Held-out N values:", held_out_N_values)
print("Input shape:", features.shape)  # (batch, 209, 3)
print("Target shape:", targets.shape)
print("Split sizes:", len(train_set), len(validation_set), len(test_set))
print("Example encoded N values:", torch.unique(features[:, 0, 2]))

## Model

The only architectural change from the successful single-$N$ model is `input_size=3`; the third feature is $N/N_{\mathrm{ref}}$.

In [ ]:
torch.manual_seed(seed)

model_config = {
    "input_size": 3,
    "hidden_size": 128,
    "num_layers": 2,
    "grid_size": 401,
    "dropout": 0.1,
    "state_min": state_min,
    "state_max": state_max,
    "n_reference": n_reference,
}
model = ConditionedTrajectoryGRUGrid(**model_config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5,
)

print(model)
print("Grid spacing:", (state_max - state_min) / (model.grid_size - 1))
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

## Train and save the best validation checkpoint

In [ ]:
epochs = 100
best_validation_loss = float("inf")
train_losses = []
validation_losses = []
checkpoint_path = experiment_dir / "withN_grugrid_best.pt"

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    total_targets = 0

    for features, next_sz in train_loader:
        features = features.to(device, non_blocking=True)
        next_sz = next_sz.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(features)
        loss = interpolated_grid_nll(
            next_sz,
            logits,
            state_min=model.state_min,
            state_max=model.state_max,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        target_count = next_sz.numel()
        total_loss += loss.item() * target_count
        total_targets += target_count

    train_loss = total_loss / total_targets
    validation_loss = evaluate(model, validation_loader, device)
    train_losses.append(train_loss)
    validation_losses.append(validation_loss)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save(
            {
                "epoch": epoch,
                "model_type": "ConditionedTrajectoryGRUGrid",
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_grid_nll": validation_loss,
                "model_config": model_config,
                "time_end": time_end,
                "seed": seed,
                "fit_n_values": fit_N_values,
                "holdout_n_values": held_out_N_values,
                "split_indices": {
                    "train": train_indices,
                    "validation": validation_indices,
                    "test": test_indices,
                },
                "source_files": {
                    str(N): source_files[N] for N in all_N_values
                },
            },
            checkpoint_path,
        )

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d}/{epochs} | "
            f"train grid NLL: {train_loss:.5f} | "
            f"validation grid NLL: {validation_loss:.5f}"
        )

print(f"Best validation grid NLL: {best_validation_loss:.5f}")
print("Saved checkpoint:", checkpoint_path)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(validation_losses, label="validation")
plt.xlabel("Epoch")
plt.ylabel("Categorical grid NLL")
plt.legend()
plt.tight_layout()
plt.show()

## Reload and evaluate each trained $N$ separately

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model = ConditionedTrajectoryGRUGrid(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Best checkpoint epoch:", checkpoint["epoch"])
print(f"Aggregate seen-N test NLL: {evaluate(model, test_loader, device):.5f}")
print()
for N in fit_N_values:
    indices_for_N = [
        index
        for index in test_indices
        if int(dataset.particle_numbers[index].item()) == N
    ]
    loader_for_N = DataLoader(
        Subset(dataset, indices_for_N),
        batch_size=batch_size,
        shuffle=False,
        pin_memory=device.type == "cuda",
    )
    print(f"N={N:2d} test grid NLL: {evaluate(model, loader_for_N, device):.5f}")

## Optional completely held-out-$N$ evaluation

This cell is active when `held_out_N_values` is nonempty. No trajectory from those $N$ values participated in optimization or checkpoint selection.

In [ ]:
if not held_out_N_values:
    print("No N is held out. Set held_out_N_values = [18] and rerun for an extrapolation test.")
else:
    for N in held_out_N_values:
        heldout_sz = trajectory_groups[N]
        heldout_labels = np.full(len(heldout_sz), N, dtype=np.float32)
        heldout_dataset = MultiNNextStateDataset(
            heldout_sz,
            heldout_labels,
            time_end=time_end,
            state_min=state_min,
            state_max=state_max,
            n_reference=n_reference,
        )
        heldout_loader = DataLoader(
            heldout_dataset,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=device.type == "cuda",
        )
        print(
            f"Completely held-out N={N} grid NLL: "
            f"{evaluate(model, heldout_loader, device):.5f}"
        )

        # A held-out one-step NLL is not enough: also test free rollouts.
        exact_holdout = heldout_dataset.sz.to(device)
        holdout_generator = torch.Generator(device=device).manual_seed(seed + N)
        generated_holdout = generate_trajectories(
            model,
            particle_number=N,
            initial_sz=exact_holdout[:, 0],
            n_time_points=exact_holdout.shape[1],
            time_end=time_end,
            generator=holdout_generator,
        )
        print("Held-out free-rollout summary:")
        for name, value in rollout_summary(generated_holdout, exact_holdout).items():
            print(f"  {name}: {value:.6f}")

        holdout_time = np.linspace(0.0, 70.0, exact_holdout.shape[1])
        generated_np = generated_holdout.cpu().numpy()
        exact_np = exact_holdout.cpu().numpy()
        plt.figure(figsize=(11, 4))
        plt.plot(holdout_time, generated_np.mean(0), label="generated mean")
        plt.plot(holdout_time, exact_np.mean(0), "k--", label="exact mean")
        plt.fill_between(
            holdout_time,
            np.percentile(generated_np, 10, axis=0),
            np.percentile(generated_np, 90, axis=0),
            alpha=0.25,
            label="generated 10--90%",
        )
        plt.xlabel(r"Time $t\gamma_F$")
        plt.ylabel(r"$s_z(t)$")
        plt.title(f"Completely held-out extrapolation test: N={N}")
        plt.legend()
        plt.tight_layout()
        plt.show()

## Free-running check at a known $N$

In [ ]:
comparison_N = max(fit_N_values)
comparison_indices = [
    index
    for index in test_indices
    if int(dataset.particle_numbers[index].item()) == comparison_N
]
exact_known = dataset.sz[torch.as_tensor(comparison_indices)].to(device)
known_generator = torch.Generator(device=device).manual_seed(seed + 1)
generated_known = generate_trajectories(
    model,
    particle_number=comparison_N,
    initial_sz=exact_known[:, 0],
    n_time_points=exact_known.shape[1],
    time_end=time_end,
    generator=known_generator,
)

summary = rollout_summary(generated_known, exact_known)
for name, value in summary.items():
    print(f"{name}: {value:.6f}")

physical_time = np.linspace(0.0, 70.0, exact_known.shape[1])
generated_np = generated_known.cpu().numpy()
exact_np = exact_known.cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(physical_time, generated_np.mean(0), label="generated mean")
axes[0].plot(physical_time, exact_np.mean(0), "k--", label="exact mean")
axes[0].set_ylabel(r"Mean $s_z(t)$")
axes[0].legend()

axes[1].fill_between(
    physical_time,
    np.percentile(generated_np, 10, axis=0),
    np.percentile(generated_np, 90, axis=0),
    alpha=0.3,
    label="generated 10--90%",
)
axes[1].plot(physical_time, np.percentile(exact_np, 10, axis=0), "k--")
axes[1].plot(physical_time, np.percentile(exact_np, 90, axis=0), "k--", label="exact 10--90%")
axes[1].set_xlabel(r"Time $t\gamma_F$")
axes[1].set_ylabel(r"$s_z(t)$")
axes[1].legend()
fig.suptitle(f"Known-N rollout check: N={comparison_N}")
plt.tight_layout()
plt.show()

## Generate at a larger, unseen $N$

This performs numerical extrapolation. Boundedness is guaranteed, but physical accuracy is not; first inspect the held-out-$N$ diagnostic above.

In [ ]:
target_N = 20
n_generated = 500
initial_sz = torch.full((n_generated,), -1.0, dtype=torch.float32)
extrapolation_generator = torch.Generator(device=device).manual_seed(123)

generated_larger_N = generate_trajectories(
    model,
    particle_number=target_N,
    initial_sz=initial_sz,
    n_time_points=next(iter(trajectory_groups.values())).shape[1],
    time_end=time_end,
    temperature=1.0,
    generator=extrapolation_generator,
).cpu().numpy()

print("Generated shape:", generated_larger_N.shape)
print("Generated range:", generated_larger_N.min(), generated_larger_N.max())
print("Conditioning feature N/N_ref:", target_N / n_reference)

physical_time = np.linspace(0.0, 70.0, generated_larger_N.shape[1])
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for trajectory in generated_larger_N[:6]:
    axes[0].plot(physical_time, trajectory, lw=1.2, alpha=0.75)
axes[0].set_ylabel(r"$s_z(t)$")
axes[0].set_title(f"Example extrapolated trajectories, N={target_N}")

axes[1].plot(
    physical_time,
    generated_larger_N.mean(axis=0),
    lw=2,
    label="generated mean",
)
axes[1].fill_between(
    physical_time,
    np.percentile(generated_larger_N, 10, axis=0),
    np.percentile(generated_larger_N, 90, axis=0),
    alpha=0.3,
    label="generated 10--90%",
)
axes[1].set_xlabel(r"Time $t\gamma_F$")
axes[1].set_ylabel(r"$s_z(t)$")
axes[1].legend()
plt.tight_layout()
plt.show()

## Pooled state distribution at the extrapolated $N$

In [ ]:
time_mask = physical_time >= 0
generated_values = generated_larger_N[:, time_mask].ravel()

plt.figure(figsize=(8, 4.5))
plt.hist(
    generated_values,
    bins=np.linspace(-1, 1, 81),
    density=True,
    histtype="stepfilled",
    color="tab:blue",
    alpha=0.45,
    linewidth=2,
    label=f"generated N={target_N}",
)
plt.xlabel(r"$s_z$")
plt.ylabel("Probability density")
plt.legend()
plt.tight_layout()
plt.show()